# pyfixest regression + MLflow tracking

A short end-to-end example: make some data, run a couple of regressions through `regress` (which wraps a pyfixest model and logs it to MLflow), then pull the logged runs back with `coeftable` and `etable`.

In [ ]:
import warnings

# keep outputs reproducible: this warning embeds an absolute path
warnings.filterwarnings("ignore", message="IProgress not found")

import logging
import os

# artifact-download progress bars print throughput rates -> not reproducible
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import mlflow
import numpy as np
import pandas as pd

from pyfixest_regression import coeftable, etable, regress

# mlflow's INFO logs carry timestamps; silence them so outputs are stable
logging.getLogger("mlflow").setLevel(logging.ERROR)

mlflow.set_tracking_uri("sqlite:///mlflow.db")
_ = mlflow.set_experiment("pyfixest-regression-example")

## Make some data

A treatment-effect DGP with a binary `treat` and two covariates. `age` and `income` are drawn **correlated** (cov ≈ 0.6), so dropping one shifts the other's coefficient — which makes comparing specifications interesting.

`outcome = 1 + 2·treat − 0.03·age + 0.5·income + noise`

In [ ]:
rng = np.random.default_rng(0)
n = 500

treat = rng.integers(0, 2, size=n)  # binary treatment (0/1)
# age and income are correlated (cov 6 over sds 10 and 1 -> corr ~ 0.6)
age, income = rng.multivariate_normal(
    mean=[50.0, 0.0], cov=[[100.0, 6.0], [6.0, 1.0]], size=n
).T
outcome = 1.0 + 2.0 * treat - 0.03 * age + 0.5 * income + rng.normal(size=n)

data = pd.DataFrame({"outcome": outcome, "treat": treat, "age": age, "income": income})
data.head()

,outcome,treat,age,income
0,0.983391,1,48.026230,-0.424792
1,1.259387,1,24.491673,-1.799903
2,4.325008,1,62.202373,0.898371
3,0.961788,0,50.337020,0.873421
4,-0.951984,0,59.177499,1.197965


## Run some regressions

Two specifications on the same data, each given a `name`. The second adds `income`; because `income` is correlated with `age`, the `age` coefficient moves between them. `regress` returns the fitted pyfixest model and logs everything to MLflow.

In [ ]:
fit_short = regress("outcome ~ treat + age", data=data, name="short")
fit_full = regress("outcome ~ treat + age + income", data=data, name="full")

fit_full.summary()

###

Estimation:  OLS
Dep. var.: outcome
sample: None = all
Inference:  iid
Observations:  500

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Intercept     |      1.301 |        0.310 |     4.201 |      0.000 |  0.692 |   1.909 |
| treat         |      1.948 |        0.094 |    20.782 |      0.000 |  1.763 |   2.132 |
| age           |     -0.035 |        0.006 |    -5.895 |      0.000 | -0.046 |  -0.023 |
| income        |      0.478 |        0.060 |     7.963 |      0.000 |  0.360 |   0.596 |
---
RMSE: 1.033 R2: 0.502 


## Every coefficient across the runs

`coeftable` stacks each run's coefficients into one long, self-describing frame (one row per run × coefficient), with the formula and the run's params joined on. Columns use plain snake_case names, including the 95% CI (`ci_low`/`ci_high`).

In [ ]:
coefs = coeftable("pyfixest-regression-example")
coefs[["fml", "coefficient", "estimate", "std_error", "p_value", "ci_low", "ci_high"]]

,fml,coefficient,estimate,std_error,p_value,ci_low,ci_high
0,outcome ~ treat + age + income,Intercept,1.300891,0.309679,3.154480e-05,0.692446,1.909336
1,outcome ~ treat + age + income,treat,1.947587,0.093715,0.000000e+00,1.763458,2.131715
2,outcome ~ treat + age + income,age,-0.034867,0.005915,6.900000e-09,-0.046488,-0.023245
3,outcome ~ treat + age + income,income,0.478348,0.060069,0.000000e+00,0.360327,0.596369
4,outcome ~ treat + age,Intercept,-0.103121,0.270103,7.027850e-01,-0.633806,0.427564
5,outcome ~ treat + age,treat,1.931537,0.099403,0.000000e+00,1.736235,2.126839
6,outcome ~ treat + age,age,-0.007105,0.005069,1.616716e-01,-0.017065,0.002855


## Side-by-side regression table

`etable` rebuilds a cross-run comparison — one column per run, headed by the run's `name` (or an abbreviated formula when it has none). Note how the `age` coefficient shifts once the correlated `income` is added.

In [ ]:
etable("pyfixest-regression-example")

,short,full
Intercept,-0.103 (0.270),1.301*** (0.310)
treat,1.932*** (0.099),1.948*** (0.094)
age,-0.007 (0.005),-0.035*** (0.006)
income,,0.478*** (0.060)
fml,outcome ~ treat + age,outcome ~ treat + age + income
nobs,500,500
r2,0.439,0.502
adj_r2,0.436,0.499
